In [1]:
RHO = 0.10                # <-- change per run: 0.10, 0.30, 0.40, or 0.50
INIT_TYPE = 'dinov3'       # <-- change per run: 'dinov3' or 'coco'

In [2]:
!pip install ultralytics -q
from ultralytics import YOLO
import torch, pandas as pd, os

def mem():
    return f"{torch.cuda.memory_allocated()/1e9:.2f} GB"
print("Baseline:", mem())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Baseline: 0.00 GB


In [3]:
CONFIG = {
    'architecture': 'yolo12m',
    'partition_csv': '/kaggle/input/datasets/redwanahmed2025/groupj-partb-partition/partition.csv',
    'label_fraction': RHO,
    'seed': 42,
    'epochs': 50,
    'img_size': 640,
    'batch_size': 32,
    'lr': 1e-3,
}
print(CONFIG)

{'architecture': 'yolo12m', 'partition_csv': '/kaggle/input/datasets/redwanahmed2025/groupj-partb-partition/partition.csv', 'label_fraction': 0.1, 'seed': 42, 'epochs': 50, 'img_size': 640, 'batch_size': 32, 'lr': 0.001}


In [4]:
partition = pd.read_csv(CONFIG['partition_csv'])

def build_rho_subset(rho, partition_df):
    k = int(round(rho * 10))
    return partition_df[(partition_df['role'] == 'pool') & (partition_df['pool_fold'] < k)]

d_rho_df = build_rho_subset(RHO, partition)
print(f"D_{RHO} size:", len(d_rho_df))

test_val_ids = set(partition[partition['role'].isin(['test', 'val'])]['image_id'])
overlap = test_val_ids.intersection(set(d_rho_df['image_id']))
print("Overlap (must be 0):", len(overlap))

D_0.1 size: 1000
Overlap (must be 0): 0


In [5]:
def build_yolo_split(df, split_name, base_dir='/kaggle/working/yolo_data'):
    img_dir = f'{base_dir}/{split_name}/images'
    lbl_dir = f'{base_dir}/{split_name}/labels'
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    for _, row in df.iterrows():
        img_src, lbl_src = row['image_path'], row['label_path']
        img_dst = f"{img_dir}/{os.path.basename(img_src)}"
        lbl_dst = f"{lbl_dir}/{os.path.basename(lbl_src)}"
        if not os.path.exists(img_dst):
            os.symlink(img_src, img_dst)
        if not os.path.exists(lbl_dst) and os.path.exists(lbl_src):
            os.symlink(lbl_src, lbl_dst)
    return img_dir, lbl_dir

val_df = partition[partition['role'] == 'val']
test_df = partition[partition['role'] == 'test']
build_yolo_split(d_rho_df, 'train')
build_yolo_split(val_df, 'val')
build_yolo_split(test_df, 'test')

print("Train:", len(os.listdir('/kaggle/working/yolo_data/train/images')))
print("Val:", len(os.listdir('/kaggle/working/yolo_data/val/images')))
print("Test:", len(os.listdir('/kaggle/working/yolo_data/test/images')))

data_yaml_content = """
path: /kaggle/working/yolo_data
train: train/images
val: val/images
test: test/images
names:
  0: Cavities
  1: Damage
  2: Infection
  3: Wisdom
"""
with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(data_yaml_content)

Train: 1000
Val: 1000
Test: 1000


In [6]:
if INIT_TYPE == 'dinov3':
    checkpoint_dir = '/kaggle/input/datasets/kanizmithila/groupj-partb-dinov3-checkpoint'  # adjust if path differs
    dinov3_state = torch.load(f'{checkpoint_dir}/dinov3_backbone.pt', map_location='cpu')

    class YOLOv12Backbone(torch.nn.Module):
        def __init__(self, full_model):
            super().__init__()
            self.layers = torch.nn.Sequential(*[full_model.model.model[i] for i in range(9)])
        def forward(self, x):
            return self.layers(x)

    fresh_model = YOLO('yolo12m.yaml')
    remapped_state = {k.replace('layers.', 'model.', 1): v for k, v in dinov3_state.items()}
    missing, unexpected = fresh_model.model.load_state_dict(remapped_state, strict=False)
    print(f"Unexpected keys (should be 0): {len(unexpected)}")
    fresh_model.save('/kaggle/working/model_initialized.pt')
    model = YOLO('/kaggle/working/model_initialized.pt')

elif INIT_TYPE == 'coco':
    model = YOLO('yolo12m.pt')  # standard COCO-pretrained weights

print(f"Model ready: {INIT_TYPE} @ rho={RHO}")

Unexpected keys (should be 0): 0
Model ready: dinov3 @ rho=0.1


In [7]:
run_name = f'bonus_{INIT_TYPE}_rho{int(RHO*100)}'
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=CONFIG['epochs'], imgsz=CONFIG['img_size'], batch=CONFIG['batch_size'],
    lr0=CONFIG['lr'], seed=CONFIG['seed'],
    project='/kaggle/working/runs', name=run_name, exist_ok=True, patience=0,
)

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/model_initialized.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bonus_dino

In [8]:
best_model = YOLO(f'/kaggle/working/runs/{run_name}/weights/best.pt')
test_metrics = best_model.val(
    data='/kaggle/working/data.yaml', split='test',
    imgsz=CONFIG['img_size'], batch=CONFIG['batch_size'],
    project='/kaggle/working/runs', name=f'{run_name}_test_eval', exist_ok=True,
)
print(f"\n=== {INIT_TYPE.upper()} @ rho={RHO} — TEST SET ===")
print(f"mAP50: {test_metrics.box.map50:.4f} | mAP50-95: {test_metrics.box.map:.4f}")
print(f"Precision: {test_metrics.box.mp:.4f} | Recall: {test_metrics.box.mr:.4f}")

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO12m summary (fused): 169 layers, 20,107,996 parameters, 0 gradients, 70.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 7.5±2.0 MB/s, size: 37.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/working/yolo_data/test/labels... 1000 images, 31 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 288.7it/s 3.5s
val: New cache created: /kaggle/working/yolo_data/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 1.4s/it 44.6s
                   all       1000       2456      0.426      0.491      0.419       0.18
              Cavities        315        662      0.335      0.453      0.325      0.114
                Damage        354       1060      0.361      0.384      0.291     0.